# Week 8 · Harnesses and Agentic Coding

**The problem.** Lakeshore Ride wants its tools (handbook search, fares, stations) usable by *any* agent: a partner's chatbot, an employee's Copilot, your own agents. And riders at a full station need to know where to go next. You publish the tools over MCP, then have a coding agent build the new `nearest_station_with_docks` tool from a specification written as tests, with you reviewing every change.

An agent is a model plus a **harness**: the loop that calls tools, the context it is given, the permissions it has, and the checks that tell it whether it succeeded. Today you work on both sides of that idea.

| Part | What you do | Lecture notes |
|---|---|---|
| 1 | Anatomy of a harness | §1, §2 |
| 2 | Run the service locally; speak MCP by hand | §4, §5 |
| 3 | Your tools in other agents: MCP client, Agent Framework, VS Code | §7, §9 |
| 4 | **Your turn:** agentic coding, a feature from a spec written as tests | §8, §13, §14 |
| 5 | Deploy and score | |
| 6 | Write your own check | §1 |

**You ship:** `POST /mcp` with a new tool, `nearest_station_with_docks`. **You submit:** your URL.

**How to use this notebook.** Run the cells in order. Parts 1 to 3 run your MCP server locally and connect three kinds of client to it; Part 4 is where a coding agent changes the service and you review it; Parts 5 and 6 ship it and grade it. The [lecture notes](lecture.md) explain the why behind each part.

**Optional, for depth:** [deep-dive.ipynb](deep-dive.ipynb) goes under the hood: MCP errors and version checks, tool schemas and tool poisoning, Agent Framework's harness agent with skills, a Foundry agent using a remote MCP server, and the context a coding agent reads in this repository. It is not graded and cannot change your score; open it after you submit.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course.py").exists())))
import course

course.setup()

## Part 1 · Anatomy of a harness

You have already built every part of a harness. This week you name them.

| Part | What it does | Where you met it |
|---|---|---|
| **Loop** | Call the model, run the tools it asks for, feed results back, stop when done | Week 6: the tool loop |
| **Tools** | The actions and data the agent can reach | `app/tools.py`, and today MCP |
| **Context** | What the model sees: instructions, retrieved passages, history | Weeks 5-6: instructions, RAG |
| **Permissions** | Which tools each agent may call, and what needs approval | Week 7: least privilege, approvals |
| **Verification** | How the agent (or you) knows the work is right | The course harness, tests |

A coding agent is the same machine pointed at a repository: its tools read files, edit them and run commands; its context includes files like `AGENTS.md`; its verification is your test suite. The better your tests and instructions, the better the agent works. That is the lesson of Part 4.

```mermaid
flowchart LR
    M["Model"] -- "tool call" --> L["Loop"]
    L -- "runs" --> T["Tools: your functions, MCP servers, shell"]
    T -- "result" --> L
    L -- "context: instructions, files, history" --> M
    L -- "done?" --> V["Verification: tests, harness"]
    V -- "not yet" --> L
```

**Two meanings of "harness".** The table describes an **agent harness**: the program that runs the model. The course harness you have used since Week 5 is an **evaluation harness**: it runs tasks against your service and grades the outcome. Anthropic's [Demystifying evals for AI agents](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents) uses the same distinction. Agent Framework ships a ready-made agent harness, [`create_harness_agent`](https://learn.microsoft.com/en-us/agent-framework/concepts/harness), which adds plans, context compaction, file memory and approvals around an agent.

## Part 2 · Run the service locally

Until now you scored the service in-process or in Azure. A real local server lets other programs connect to it: an MCP client, VS Code, a browser. This starts one in the background, on port 8000.

In [ ]:
import os
import socket
import subprocess
import time

import httpx2

LOCAL = "http://127.0.0.1:8000"
with socket.socket() as probe:
    if probe.connect_ex(("127.0.0.1", 8000)) == 0:
        raise RuntimeError("Port 8000 is busy. Stop the other server (or restart this kernel) and run the cell again.")

# The local server runs without an API key; Week 10 adds one to the deployed service.
env = {**os.environ, "SERVICE_API_KEY": ""}
server = subprocess.Popen([sys.executable, "-m", "uvicorn", "app.main:app", "--port", "8000"], cwd=course.FY26, env=env)

for _ in range(60):
    try:
        if httpx2.get(f"{LOCAL}/health", timeout=2).status_code == 200:
            print(f"Service running at {LOCAL} (open {LOCAL}/docs in a browser)")
            break
    except httpx2.HTTPError:
        time.sleep(1)

### MCP by hand

MCP over HTTP is plain JSON-RPC: you POST `{"jsonrpc": "2.0", "id": ..., "method": ..., "params": ...}` to one URL. Since revision [2026-07-28](https://modelcontextprotocol.io/specification/2026-07-28/changelog) the protocol is **stateless**: no handshake, no session. Every request names its protocol version and its client in `params._meta`, and repeats the essentials in HTTP headers (`MCP-Protocol-Version`, `Mcp-Method`, `Mcp-Name`) so that gateways can route and log it without reading the body.

Three methods matter most: `server/discover` (who are you, what do you support?), `tools/list` (what can you do?) and `tools/call` (do it). Before using an SDK, see the raw messages.

In [ ]:
import json

PROTOCOL = "2026-07-28"
META = {
    "io.modelcontextprotocol/protocolVersion": PROTOCOL,
    "io.modelcontextprotocol/clientInfo": {"name": "notebook", "version": "1"},
    "io.modelcontextprotocol/clientCapabilities": {},
}


def rpc(method: str, params: dict | None = None) -> dict:
    params = {**(params or {}), "_meta": META}
    headers = {"Accept": "application/json, text/event-stream", "MCP-Protocol-Version": PROTOCOL, "Mcp-Method": method}
    if "name" in params:
        headers["Mcp-Name"] = params["name"]  # tools/call also names the tool in a header
    body = {"jsonrpc": "2.0", "id": 1, "method": method, "params": params}
    return httpx2.post(f"{LOCAL}/mcp", json=body, headers=headers, timeout=60).json()


about = rpc("server/discover")["result"]
print("server:", about["_meta"]["io.modelcontextprotocol/serverInfo"]["name"], "| versions:", about["supportedVersions"])

tools = rpc("tools/list")["result"]["tools"]
for tool in tools:
    print(f"\n{tool['name']}: {tool['description']}")
print("\ninput schema of quote_fare:", json.dumps(next(t for t in tools if t["name"] == "quote_fare")["inputSchema"], indent=2))

In [ ]:
answer = rpc("tools/call", {"name": "quote_fare", "arguments": {"plan": "member", "minutes": 50, "ebike": False}})
print(json.dumps(answer["result"], indent=2))

The descriptions and schemas are generated from the Python functions in `app/mcp_server.py`: their names, type hints and docstrings. That text is all a client agent knows about your tools, so write it as carefully as a public API.

Why stateless? Because any replica of your Container App can now answer any request: no sticky sessions, no session store. Earlier revisions (up to 2025-11-25) opened a session with an `initialize` handshake and an `Mcp-Session-Id` header. The MCP Python SDK serves both on the same endpoint, so older clients keep working. See [Versioning and compatibility](https://modelcontextprotocol.io/specification/2026-07-28/basic/versioning).

## Part 3 · Your tools, in other agents

### With the official MCP client

The SDK's `Client` speaks the protocol for you. Given a URL, it tries the stateless revision first and falls back to the handshake for older servers.

In [ ]:
from mcp import Client

async with Client(f"{LOCAL}/mcp") as mcp_client:
    listed = await mcp_client.list_tools()
    print("tools:", [t.name for t in listed.tools])
    result = await mcp_client.call_tool("get_station", {"station_id": "LS-08"})
    print("LS-08:", result.structured_content)

### Inside an Agent Framework agent

Here the agent lives in this notebook, and its tools live in your service. The agent discovers them over MCP at start-up. Add a second MCP server, Microsoft Learn's public one, and the same agent can combine your business data with documentation.

In [ ]:
from agent_framework import Agent, MCPStreamableHTTPTool

from app.config import get_settings
from app.state import AppState

client = AppState(get_settings()).models.agent_client()

async with MCPStreamableHTTPTool(name="lakeshore", url=f"{LOCAL}/mcp") as lakeshore:
    agent = Agent(
        client=client,
        name="concierge",
        instructions="You help Lakeshore Ride riders. Use the lakeshore tools for prices, stations and policies.",
        tools=[lakeshore],
    )
    result = await agent.run("I'm a student. What does a 75-minute classic ride cost, and how many e-bikes are at LS-08?")
    print(result.text)

In [ ]:
async with (
    MCPStreamableHTTPTool(name="lakeshore", url=f"{LOCAL}/mcp") as lakeshore,
    MCPStreamableHTTPTool(name="learn", url="https://learn.microsoft.com/api/mcp") as learn,
):
    agent = Agent(client=client, name="concierge",
                  instructions="Use lakeshore tools for Lakeshore Ride facts and learn tools for Microsoft documentation. Be brief.",
                  tools=[lakeshore, learn])
    result = await agent.run("What is the lost e-bike fee at Lakeshore Ride? And in one sentence, what is Azure Container Apps?")
    print(result.text)

### In VS Code

Your coding agent can use the same tools.

1. Open the Command Palette (`Cmd+Shift+P` / `Ctrl+Shift+P`) and run **MCP: Add Server**.
2. Choose **HTTP**, enter `http://127.0.0.1:8000/mcp`, and name it `lakeshore`.
3. In Copilot Chat, agent mode, ask: *"Use the lakeshore tools: what does a 50-minute classic ride cost for a member?"*
4. Expand the tool call in the chat and check that it came from `lakeshore`.

Later, point it at your deployed URL plus `/mcp`. See [Add MCP servers in VS Code](https://code.visualstudio.com/docs/copilot/customization/mcp-servers).

Stop the local server when you are done with this part:

In [ ]:
server.terminate()
try:
    server.wait(timeout=10)
except subprocess.TimeoutExpired:
    server.kill()  # a hard stop, if the server did not exit in time
print("local server stopped")

## Part 4 · Agentic coding: a feature from a spec

Riders at a full station need to know where to go. You will add a tool, `nearest_station_with_docks`, and you will not write it yourself: a coding agent will. Your job is the part agents cannot do for you: a precise specification, a good prompt, and a careful review.

### The specification is a test file

Read `tests/test_week08_nearest_station.py`. It states the behavior exactly, including two traps: a full station that is closer, and a case where flat latitude/longitude distance gives the wrong answer. Right now its tests are **skipped**, because the function does not exist yet:

In [ ]:
print(Path("tests/test_week08_nearest_station.py").read_text()[:1200])
course.run([sys.executable, "-m", "pytest", "tests/test_week08_nearest_station.py", "-q", "-rs"], check=False)

### The instructions are `AGENTS.md`

Coding agents read `AGENTS.md` before they start: commands, rules, the definition of done. It is to an agent what onboarding notes are to a new teammate. Two lessons from teams that run coding agents at scale:

- **Keep it short: a map, not an encyclopedia.** Point to where things live; do not paste them in. Every line competes for the agent's attention ([Harness engineering, OpenAI](https://openai.com/index/harness-engineering/)).
- **Tell it how to verify its work.** An agent that can run the tests checks itself; one that cannot will report success anyway ([Claude Code best practices](https://code.claude.com/docs/en/best-practices)).

In [ ]:
print(Path("AGENTS.md").read_text())

### Your turn

Open your coding agent (Copilot Chat in **agent mode**, Claude Code, Codex, or another agent) with the `fy26` folder as context, and give it a prompt like this one. Adjust it; a good prompt is specific about the goal, the constraints and the proof.

> Read `AGENTS.md` and `tests/test_week08_nearest_station.py`. Implement `nearest_station_with_docks` in `app/domain.py` so that every test in that file passes, using the existing `distance_km`. Publish it as an MCP tool with the same name in `app/mcp_server.py`, with a docstring a client agent can understand. Do not modify any test. When you are done, run `uv run pytest` and `uv run ruff check .` and show me the output.

While it works, watch the loop from Part 1: it reads files, edits, runs tests, reads failures, edits again. When it finishes, **review the diff before you accept it**:

- Does it skip the origin station, closed stations and stations with zero free docks?
- Does it use `distance_km` (great-circle), not raw latitude and longitude?
- Does an unknown id return `{"error": ...}` instead of raising?
- Did it change anything it was not asked to change?

**If the agent gets stuck,** give it the exact failing test output and ask it to fix that test only. If two rounds do not converge, stop and read the failing assertion yourself: usually the prompt was ambiguous, and the fix is a sentence in the prompt, not more code. If some tests stay red, ship what passes and move on to Part 5; a published tool with some correct answers still scores.

Then verify with your own tools, not the agent's word:

In [ ]:
course.run([sys.executable, "-m", "pytest", "-q"], check=False)
proof = course.check(8, in_process=True)

`w08.mcp.new_tool` should now pass. If it does not, give the agent the failing check's detail line and the proof file, and let it iterate. That is the loop again, with you supplying the verification.

## Part 5 · Deploy and score

In [ ]:
url = course.deploy()
proof = course.check(8, url)
print(f"\nMCP endpoint for VS Code or any agent: {url}/mcp")

Submit your URL by the deadline in the [Week 8 README](README.md).

## Part 6 · Write your own check

A harness is only as good as its checks. The course checks test the happy path of your new tool. Write one for the error case. A check takes a `Context` and returns `(score, detail)`; `call_tool` speaks MCP for you.

In [ ]:
from fastapi.testclient import TestClient

from app.main import create_app
from harness.checks.week08 import MCPError, call_tool
from harness.client import Target
from harness.engine import Context


def unknown_station_is_an_error(ctx: Context) -> tuple[float, str]:
    try:
        result = call_tool(ctx, "nearest_station_with_docks", {"station_id": "LS-99"})
    except MCPError as exc:  # a check turns a failure into a score and a reason; it never crashes the run
        return 0.0, str(exc)
    ok = isinstance(result, dict) and "error" in result
    return (1.0 if ok else 0.0), f"LS-99 -> {result}"


with TestClient(create_app()) as test_client:
    target = Target(client=test_client, api_key=os.environ.get("SERVICE_API_KEY"))
    print(unknown_station_is_an_error(Context(target=target, week=8)))

Instructors add checks like this one to the private harness. Now you know how.

### What makes a good check

- **Grade the outcome, not the path.** Check the answer the tool returns, not which helper produced it.
- **Separate capability from regression.** A capability check asks "can it do the new thing?"; a regression check asks "does the old thing still work?". Week 10 has both.
- **One run is one sample.** A model-backed service is not deterministic. *pass@k* asks whether at least one of *k* runs succeeds; *pass^k* asks whether all *k* do. Your users experience pass^k.
- **Read the transcripts.** The proof in `reports/` records every request and response. A score tells you *that* something failed; the transcript tells you *why*.

These ideas come from [Demystifying evals for AI agents (Anthropic)](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents).

### Go further

- Connect VS Code to your **deployed** `/mcp` (after Week 10, add the `X-API-Key` header in the server's settings).
- Publish the handbook pages as MCP **resources** (`@mcp.resource("handbook://{doc_id}")`), so clients can read a page without calling a model.
- Ask your coding agent to add the error-case check above to `harness/checks/week08.py`. Review it as you would a colleague's pull request.
- Read [Code execution with MCP (Anthropic)](https://www.anthropic.com/engineering/code-execution-with-mcp): when an agent has hundreds of tools, it can call them from code instead of loading every schema into its context.
- **Deep dive (optional):** [deep-dive.ipynb](deep-dive.ipynb) speaks MCP's error paths by hand, shows why an `enum` beats a docstring, runs a tool-poisoning attack safely, builds a harness agent with a skill, and counts the tokens this repository's instructions cost a coding agent.

### What to remember

- An agent is a model plus a harness: loop, tools, context, permissions, verification.
- MCP makes your tools available to any agent; their names, schemas and docstrings are their documentation.
- Current MCP is stateless: every request carries its version and client, so any replica can serve it.
- Run locally before you deploy: it is the fastest feedback loop you have.
- With a coding agent, *you* own the specification, the instructions and the review. Tests are the best specification an agent can receive.
- Trust the tests and the harness, not the agent's summary of its own work.

**Learn more:** [Effective harnesses for long-running agents](https://www.anthropic.com/engineering/effective-harnesses-for-long-running-agents) · [What is MCP?](https://modelcontextprotocol.io/docs/getting-started/intro) · [MCP Python SDK](https://py.sdk.modelcontextprotocol.io/) · [Building Applications with GitHub Copilot Agent Mode](https://learn.microsoft.com/en-us/training/modules/github-copilot-agent-mode/) · [AGENTS.md](https://agents.md)